# BidCoin Generation

## 할 것
1. mock retrieval 결과 불러오기
2. context / history 확인
3. prompt 확인
4. OpenAI 호출
5. 답변 품질 실험


In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

print("project_root =", project_root)
print("src_path =", src_path)

project_root = C:\develop_project\codeit-4team-2nd_project-Bidcoin\Bidcoin
src_path = C:\develop_project\codeit-4team-2nd_project-Bidcoin\Bidcoin\src


In [2]:
from generation.mock_data import get_mock_retrieval_result
from generation.context_builder import build_context_block, build_history_block
from generation.prompts import SYSTEM_PROMPT, build_user_prompt
from generation.generator import BidCoinGenerator
from generation.config import Settings

## 1) mock data 확인

In [3]:
retrieval_result = get_mock_retrieval_result()
retrieval_result

RetrievalResult(question='콘텐츠 관리 요구사항과 보안 요구사항을 정리해줘.', contexts=[RetrievedContext(chunk_id='doc_001_chunk_01', text='본 사업은 이러닝 시스템 기능 고도화를 목적으로 한다. 학습 콘텐츠 등록, 수정, 버전관리 기능을 제공해야 하며, 관리자는 학습 이력과 진도 현황을 조회할 수 있어야 한다.', source_file='국민연금공단_이러닝시스템.hwp', organization='국민연금공단', project_name='이러닝시스템 구축', summary='이러닝 콘텐츠 관리 및 학습 이력 조회 기능을 포함한 시스템 고도화 사업', score=0.93), RetrievedContext(chunk_id='doc_001_chunk_02', text='보안 요구사항으로는 사용자 권한 분리, 개인정보보호, 로그 관리 및 관리자 행위 추적 기능이 요구된다.', source_file='국민연금공단_이러닝시스템.hwp', organization='국민연금공단', project_name='이러닝시스템 구축', summary='보안 및 개인정보보호 요구사항 포함', score=0.89), RetrievedContext(chunk_id='doc_001_chunk_03', text='운영 측면에서는 장애 대응 체계, 백업 및 복구 방안, 운영 매뉴얼 제공이 요구된다.', source_file='국민연금공단_이러닝시스템.hwp', organization='국민연금공단', project_name='이러닝시스템 구축', summary='운영 및 유지보수 요구사항 포함', score=0.84)], chat_history=[ChatTurn(role='user', content='국민연금공단 사업 문서를 찾아줘.'), ChatTurn(role='assistant', content='국민연금공단 이러닝시스템 구축 관련 문서를 참고하겠습니다.')])

## 2) context block / history block 만들기

In [ ]:
settings = Settings()
context_block, used_sources = build_context_block(
    contexts=retrieval_result.contexts,
    max_contexts=settings.max_contexts,
    max_chars=settings.max_context_chars,
)
history_block = build_history_block(retrieval_result.chat_history)

print("USED SOURCES:", used_sources)
print("\n===== CONTEXT BLOCK =====\n")
print(context_block)
print("\n===== HISTORY BLOCK =====\n")
print(history_block)

TypeError: build_history_block() got an unexpected keyword argument 'max_turns'

## 3) 실제 user prompt 확인


In [ ]:
user_prompt = build_user_prompt(
    question=retrieval_result.question,
    context_block=context_block,
    history_block=history_block,
)

print("===== SYSTEM PROMPT =====\n")
print(SYSTEM_PROMPT)
print("\n===== USER PROMPT =====\n")
print(user_prompt)

===== SYSTEM PROMPT =====

당신은 B2G 공공입찰 전문 컨설팅 어시스턴트입니다.
반드시 주어진 RFP 문서 컨텍스트만을 근거로 답변하세요.

규칙:
1. 문서에 없는 내용은 추측하지 말고 "해당 문서에서 확인할 수 없습니다."라고 답하세요.
2. 답변은 한국어로 작성하세요.
3. 답변은 가능한 한 구조화된 형식으로 작성하세요.
4. 핵심 요구사항, 목적, 일정/운영/보안 관련 내용이 보이면 구분해서 정리하세요.
5. 마지막에 반드시 "출처" 섹션을 넣고 파일명을 나열하세요.
6. 여러 문서를 비교할 때는 공통점/차이점을 분리해서 정리하세요.
7. 같은 내용을 반복하지 말고, 문서 근거가 있는 내용만 간결하게 설명하세요.

===== USER PROMPT =====

[참고 문서]
[문서 1] chunk_id=doc_001_chunk_01 | score=0.9300 | 기관=국민연금공단 | 사업명=이러닝시스템 구축 | 파일명=국민연금공단_이러닝시스템.hwp
문서 요약: 이러닝 콘텐츠 관리 및 학습 이력 조회 기능을 포함한 시스템 고도화 사업
본문:
본 사업은 이러닝 시스템 기능 고도화를 목적으로 한다. 학습 콘텐츠 등록, 수정, 버전관리 기능을 제공해야 하며, 관리자는 학습 이력과 진도 현황을 조회할 수 있어야 한다.

---

[문서 2] chunk_id=doc_001_chunk_02 | score=0.8900 | 기관=국민연금공단 | 사업명=이러닝시스템 구축 | 파일명=국민연금공단_이러닝시스템.hwp
문서 요약: 보안 및 개인정보보호 요구사항 포함
본문:
보안 요구사항으로는 사용자 권한 분리, 개인정보보호, 로그 관리 및 관리자 행위 추적 기능이 요구된다.

---

[문서 3] chunk_id=doc_001_chunk_03 | score=0.8400 | 기관=국민연금공단 | 사업명=이러닝시스템 구축 | 파일명=국민연금공단_이러닝시스템.hwp
문서 요약: 운영 및 유지보수 요구사항 포함
본문:
운영 측면에서는 장애 대응 체계, 백업 및 

## 4) OpenAI 호출

`.env`에 `OPENAI_API_KEY`가 있어야함!


In [ ]:
generator = BidCoinGenerator()
response = generator.generate(retrieval_result)

print("===== ANSWER =====\n")
print(response.answer)
print("\n===== USED SOURCES =====")
print(response.used_sources)

===== ANSWER =====

1. 요약
- 콘텐츠 관리 요구사항: 학습 콘텐츠의 등록·수정·버전관리 기능 제공, 관리자의 학습 이력 및 진도 현황 조회 기능 필요.
- 보안 요구사항: 사용자 권한 분리, 개인정보보호, 로그 관리 및 관리자 행위 추적 기능 요구.

2. 상세 내용
- 콘텐츠 관리 관련
  - 학습 콘텐츠 등록 기능: 시스템은 학습 콘텐츠를 등록할 수 있어야 함.
  - 학습 콘텐츠 수정 기능: 등록된 콘텐츠를 수정할 수 있어야 함.
  - 버전관리 기능: 콘텐츠의 버전 관리를 제공해야 함.
  - 학습 이력 조회: 관리자는 학습자의 학습 이력(학습 기록)을 조회할 수 있어야 함.
  - 진도 현황 조회: 관리자는 학습자의 진도 현황을 조회할 수 있어야 함.

- 보안 관련
  - 사용자 권한 분리: 사용자별(역할별) 권한 분리 기능을 제공해야 함.
  - 개인정보보호: 개인정보 보호 조치가 요구됨.
  - 로그 관리: 시스템 로그 관리를 수행해야 함.
  - 관리자 행위 추적: 관리자의 행위를 추적할 수 있는 기능을 제공해야 함.

3. 확인된 요구사항/근거
- 콘텐츠 관리
  - 학습 콘텐츠 등록·수정·버전관리 제공 — 근거: 문서 1 (국민연금공단_이러닝시스템.hwp, chunk_id=doc_001_chunk_01)
  - 관리자 학습 이력 및 진도 현황 조회 — 근거: 문서 1 (국민연금공단_이러닝시스템.hwp, chunk_id=doc_001_chunk_01)

- 보안
  - 사용자 권한 분리 — 근거: 문서 2 (국민연금공단_이러닝시스템.hwp, chunk_id=doc_001_chunk_02)
  - 개인정보보호 — 근거: 문서 2 (국민연금공단_이러닝시스템.hwp, chunk_id=doc_001_chunk_02)
  - 로그 관리 — 근거: 문서 2 (국민연금공단_이러닝시스템.hwp, chunk_id=doc_001_chunk_02)
  - 관리자 행위 추적 — 근거: 문서 2 (국민연금공단_이러닝시스템.hwp, chunk_id

In [ ]:
response

GenerationResponse(answer='1. 요약\n- 콘텐츠 관리 요구사항: 학습 콘텐츠의 등록·수정·버전관리 기능 제공, 관리자의 학습 이력 및 진도 현황 조회 기능 필요.\n- 보안 요구사항: 사용자 권한 분리, 개인정보보호, 로그 관리 및 관리자 행위 추적 기능 요구.\n\n2. 상세 내용\n- 콘텐츠 관리 관련\n  - 학습 콘텐츠 등록 기능: 시스템은 학습 콘텐츠를 등록할 수 있어야 함.\n  - 학습 콘텐츠 수정 기능: 등록된 콘텐츠를 수정할 수 있어야 함.\n  - 버전관리 기능: 콘텐츠의 버전 관리를 제공해야 함.\n  - 학습 이력 조회: 관리자는 학습자의 학습 이력(학습 기록)을 조회할 수 있어야 함.\n  - 진도 현황 조회: 관리자는 학습자의 진도 현황을 조회할 수 있어야 함.\n\n- 보안 관련\n  - 사용자 권한 분리: 사용자별(역할별) 권한 분리 기능을 제공해야 함.\n  - 개인정보보호: 개인정보 보호 조치가 요구됨.\n  - 로그 관리: 시스템 로그 관리를 수행해야 함.\n  - 관리자 행위 추적: 관리자의 행위를 추적할 수 있는 기능을 제공해야 함.\n\n3. 확인된 요구사항/근거\n- 콘텐츠 관리\n  - 학습 콘텐츠 등록·수정·버전관리 제공 — 근거: 문서 1 (국민연금공단_이러닝시스템.hwp, chunk_id=doc_001_chunk_01)\n  - 관리자 학습 이력 및 진도 현황 조회 — 근거: 문서 1 (국민연금공단_이러닝시스템.hwp, chunk_id=doc_001_chunk_01)\n\n- 보안\n  - 사용자 권한 분리 — 근거: 문서 2 (국민연금공단_이러닝시스템.hwp, chunk_id=doc_001_chunk_02)\n  - 개인정보보호 — 근거: 문서 2 (국민연금공단_이러닝시스템.hwp, chunk_id=doc_001_chunk_02)\n  - 로그 관리 — 근거: 문서 2 (국민연금공단_이러닝시스템.hwp, chunk_id=doc_001_chunk_02)\n  - 관리자 행위 추적 — 근거:

## 5) 질문 바꿔가며 실험

In [ ]:
custom_question = "보안 요구사항만 따로 정리해줘."

retrieval_result.question = custom_question
response = generator.generate(retrieval_result)

print(response.answer)

1. 요약
- 본 문서들에서 확인된 보안 요구사항은 사용자 권한 분리, 개인정보보호, 로그 관리 및 관리자 행위 추적 기능 등 네 가지 항목으로 요약됩니다.

2. 상세 내용
- 사용자 권한 분리
  - 시스템 내에서 사용자 권한을 분리하는 기능을 요구합니다.
- 개인정보보호
  - 개인정보보호와 관련된 요구사항이 포함되어야 합니다.
- 로그 관리
  - 시스템 운영 시 필요한 로그 관리 기능을 요구합니다.
- 관리자 행위 추적
  - 관리자의 행위에 대해 추적할 수 있는 기능을 요구합니다.

3. 확인된 요구사항/근거
- 사용자 권한 분리
  - 근거: 문서 2 (국민연금공단_이러닝시스템.hwp) — "사용자 권한 분리" 명시.
- 개인정보보호
  - 근거: 문서 2 (국민연금공단_이러닝시스템.hwp) — "개인정보보호" 명시.
- 로그 관리
  - 근거: 문서 2 (국민연금공단_이러닝시스템.hwp) — "로그 관리" 명시.
- 관리자 행위 추적
  - 근거: 문서 2 (국민연금공단_이러닝시스템.hwp) — "관리자 행위 추적 기능" 명시.

4. 출처
- 국민연금공단_이러닝시스템.hwp



## 참고
- 답변이 장황하면 프롬프트에서 항목 수 줄이기
- 출처가 빠지면 시스템 프롬프트 규칙을 더 강하게 쓰기
- hallucination이 있으면 "추론 금지"를 더 명시하기

